In [ ]:
# Set up your environment
from functools import partial
import numpy as np
import pandas as pd

import ipywidgets as widgets
import plotly.express as px
import plotly.graph_objs as go

import bql

In [ ]:
# Connect to BQL
bq = bql.Service()

In [ ]:
def get_data(universe, data_items):
    """Returns data for a universe and set of data items from BQL."""
    # Build and execute the request
    request = bql.Request(universe, data_items)
    response = bq.execute(request)
    # Process the response into a DataFrame
    df = pd.concat(
        [data_item.df()[data_item.name] for data_item in response], 
        axis=1
    )
    return df

In [ ]:
def get_options_data(ticker, minimum_open_int, expiry_cutoff):
    """Returns option data filtered by open interest and expiry for a
    chosen underlying."""
    open_int = bq.data.open_int()
    expiry = bq.data.expire_dt()
    # Filter the universe based on open interest and expiry date
    universe = (
        bq.univ.options(ticker)
        .filter(
            (open_int > minimum_open_int)
            .and_(expiry < expiry_cutoff)
        )
    )

    data_items = {
        'Open Int': open_int,
        'Volume': bq.data.px_volume(),
        'Price': bq.data.px_last(),
        'Expiry': expiry,
        'Name': bq.data.name(),
        'Put/Call': bq.data.put_call(),
        'Strike': bq.data.strike_px()
    }
    
    df = get_data(universe, data_items)
    return df

In [ ]:
# Set the default Plotly colors for Put and Call
color_discrete_map = {
    'Put': px.colors.qualitative.Plotly[1], 
    'Call': px.colors.qualitative.Plotly[0]
}

# Common annotation parameters that will be used more than once
annotation_params = {
    'annotation_text': 'Underlying Price',
    'annotation_font_size': 8, 
    'line_color': 'darkorange', 
    'annotation_font_color': 'darkorange'
}

# Set the category order so the puts appear under calls in the stacked bar
# chart
category_orders = {'Put/Call': ['Put', 'Call']}

def build_bar_fig(options_df, underlying_price):
    # Build the bar fig at startup using data from the nearest expiry
    first_date = options_df['Expiry'].min()
    bar_df = options_df[options_df['Expiry']==first_date]
    # Create the bar figure and set some styling
    bar_fig = px.bar(
        data_frame=bar_df, 
        x='Strike', 
        y='Open Int', 
        color='Put/Call', 
        color_discrete_map=color_discrete_map,
        category_orders=category_orders,
        template='plotly_dark',
        title=f'Expiry {first_date.strftime("%Y-%m-%d")}',
    )

    bar_fig.update_layout(
        margin=dict(t=30, l=10, b=10, r=10), 
        title_font_size=14,
        legend_traceorder="reversed"
    )
    # Add the line for underlying price
    bar_fig.add_vline(
        underlying_price,
        annotation_position='top', 
        line_width=1.2, 
        **annotation_params
    )

    bar_fig = go.FigureWidget(bar_fig)
    
    return bar_fig

In [ ]:
def build_bubble_fig(options_df, underlying_price):
    # Build the bubble figure
    bubble_fig = px.scatter(
        data_frame=options_df, 
        x="Expiry", 
        y="Strike",
        size="Open Int", 
        color="Put/Call",
        color_discrete_map=color_discrete_map,
        category_orders=category_orders,
        size_max=40,
        template='plotly_dark',
        width=600,
    )

    bubble_fig.update_layout(
        margin=dict(t=10, l=10, b=10, r=80),
        showlegend=False
    )
    # Add the line for underlying price
    bubble_fig.add_hline(
        underlying_price, 
        annotation_position='right', 
        line_width=0.8, 
        **annotation_params
    )
    
    bubble_fig = go.FigureWidget(bubble_fig)

    return bubble_fig

In [ ]:
# Set up listener function that will be called when the bubble chart is 
# clicked
def update_bar_chart(options_df, bar_fig, trace, points, selector):
    if len(points.xs) > 0:
        clicked_date = points.xs[0]
        # Filter the DataFrame for the chosen date and split the puts and
        # the calls
        bar_df = options_df[options_df['Expiry']==clicked_date]
        puts = bar_df[bar_df['Put/Call']=='Put']
        calls = bar_df[bar_df['Put/Call']=='Call']
        with bar_fig.batch_update():
            # Update puts
            bar_fig.data[0].y = puts['Open Int']
            bar_fig.data[0].x = puts['Strike']
            # Update calls
            bar_fig.data[1].y = calls['Open Int']
            bar_fig.data[1].x = calls['Strike']
            bar_fig.layout.title.text = f'Expiry {clicked_date}'

In [ ]:
# Input widgets
security_input = widgets.Text(
    description='Enter Security', 
    value='IBM US Equity', 
    layout={'width': '230px'},
    style={'description_width': 'initial'}
)
expiry_input = widgets.Text(
    description='Expiry Cutoff', 
    value='3M', 
    layout={'width': '150px'}
)
minimum_open_int_input = widgets.IntText(
    description='Minimum Open Interest', 
    value=0, 
    layout={'width': '195px'}, 
    style={'description_width': 'initial'}
)

# Widget to display loading status
spinner = widgets.HTML(
    '''<i class="fa fa-spinner fa-spin" style="font-size: 18px"></i>''',
    layout={'visibility': 'hidden', 'margin': '3px 0 0 10px', 'width': '40px'}
)

# Widget to initiate running of the procedure when clicked
go_button = widgets.Button(
    description='Go', 
    style={'description_width': 'initial'}, 
    layout={'width': '60px', 'margin': '3px 0 0 5px'},
    button_style='success'
)

# Empty containers to push exception message and charts into
exception_box = widgets.VBox(layout={'margin': '0 0 0 -40px'})
bubble_fig_box = widgets.VBox()
bar_fig_box = widgets.VBox()

# Collect the display widgets together
ui_display = widgets.VBox(
    [
        widgets.HBox(
            [
                security_input, 
                expiry_input, 
                minimum_open_int_input, 
                go_button, 
                spinner, 
                exception_box
            ]
        ),
        widgets.HBox(
            [bubble_fig_box, bar_fig_box]
        )
    ]
)

In [ ]:
def run(event=None):
    # Reset the display
    exception_box.children = []
    bubble_fig_box.children = []
    bar_fig_box.children = []
    spinner.layout.visibility = 'visible'
    try:
        underlying_price = get_data(
            universe=security_input.value,  
            data_items={'Underlying Price': bq.data.px_last()}
        )
        underlying_price = underlying_price['Underlying Price'].item()
        # Get the filtered option data
        options_df = get_options_data(
            ticker=security_input.value, 
            minimum_open_int=minimum_open_int_input.value,
            expiry_cutoff=expiry_input.value
        )
        # Build the bubble and bar charts
        bubble_fig = build_bubble_fig(
            options_df=options_df, 
            underlying_price=underlying_price
        )
        bar_fig = build_bar_fig(options_df, underlying_price)
        # Register the listener function
        for trace in bubble_fig.data:
            # Pass the options_df and the bar_fig to the update function so
            # that it can perform operations on them
            trace.on_click(partial(update_bar_chart, options_df, bar_fig))
        # Push the figures into the display
        bubble_fig_box.children = [bubble_fig]
        bar_fig_box.children = [bar_fig]

    except Exception as e:
        # Show the exception message
        exception_box.children = [widgets.Label(f'{e}')]
    spinner.layout.visibility = 'hidden'
    
# Register the button to the run function
go_button.on_click(run)

In [ ]:
# Run on startup
run()
# Show the output
ui_display